## 1. Locate source code and the Bank Marketing stream


In [ ]:
from pathlib import Path
import os, shutil, zipfile, json, sys, subprocess, time

INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
CODE = WORK_ROOT / "JCAM" / "Code"
RESULTS = CODE / "results" / "exp10_bank"
BOOTSTRAP_ERROR = None

def extract_matching_zip(target_name, destination):
    matches = []
    for path in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(path) as archive:
                if any(name.replace("\\", "/").endswith(target_name) for name in archive.namelist()):
                    matches.append(path)
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError(f"Expected one archive containing {target_name}, found {matches}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)

try:
    source_roots = sorted({path.parent.parent for path in INPUT_ROOT.rglob("run_application.py")
                           if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if not source_roots:
        extracted_source = WORK_ROOT / "_jcam_source"
        extract_matching_zip("Code/kaggle/run_application.py", extracted_source)
        source_roots = sorted({path.parent.parent for path in extracted_source.rglob("run_application.py")
                               if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if len(source_roots) != 1:
        raise RuntimeError(f"Expected one Code source, found {source_roots}")
    shutil.copytree(source_roots[0], CODE, dirs_exist_ok=True)
    inputs = sorted(INPUT_ROOT.rglob("bank_stream.npz"))
    if not inputs:
        extracted_data = WORK_ROOT / "_jcam_bank_input"
        extract_matching_zip("bank_stream.npz", extracted_data)
        inputs = sorted(extracted_data.rglob("bank_stream.npz"))
    if len(inputs) != 1:
        raise RuntimeError(f"Expected one bank_stream.npz, found {inputs}")
    DATA = inputs[0]
    SOURCE_META = DATA.with_suffix(".metadata.json")
    if not SOURCE_META.is_file():
        raise FileNotFoundError(SOURCE_META)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Code:", CODE, "Data:", DATA)
except Exception as error:
    BOOTSTRAP_ERROR = repr(error)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Bootstrap failed:", BOOTSTRAP_ERROR)


## 2. Validate the chronological NPZ


In [ ]:
import hashlib
import numpy as np

RUN_OK = BOOTSTRAP_ERROR is None
STAGES = []
ROUND_PATH = None

def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)

if RUN_OK:
    try:
        source = json.loads(SOURCE_META.read_text(encoding="utf-8"))
        if hashlib.sha256(DATA.read_bytes()).hexdigest() != source["npz_sha256"]:
            raise ValueError("NPZ SHA-256 mismatch")
        with np.load(DATA, allow_pickle=False) as data:
            x, y, g, r, w = [data[k] for k in ("features", "labels", "groups", "radii", "weights")]
            if x.ndim != 3 or y.shape != x.shape[:2] or g.shape != y.shape:
                raise ValueError("Invalid stream shapes")
            if r.shape != (len(x), 3) or w.shape != r.shape:
                raise ValueError("Invalid objective metadata shapes")
            if not np.all(np.isfinite(x)) or not np.all(np.isin(y, [-1, 1])):
                raise ValueError("Invalid features or labels")
            if not np.all((g == 0).any(axis=1) & (g == 1).any(axis=1)):
                raise ValueError("Both age groups are required in every batch")
            if not np.allclose(w.sum(axis=1), 1) or np.any(w < 0) or np.any(r < 0):
                raise ValueError("Invalid weights or radii")
        record("validate", "passed", f"T={len(x)}, n={x.shape[1]}, d={x.shape[2]}")
    except Exception as error:
        RUN_OK = False
        record("validate", "failed", repr(error))
else:
    record("validate", "skipped", BOOTSTRAP_ERROR)


## 3. Run EXP-10 in chronological order


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    command = [sys.executable, str(CODE / "kaggle" / "run_application.py"),
               "--data", str(DATA), "--eta", "0.05", "--output-root", str(RESULTS)]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-5000:])
    if outcome.returncode == 0:
        checkpoints = sorted((RESULTS / "application").rglob("metadata.json"))
        if len(checkpoints) != 1:
            RUN_OK = False
            record("application", "failed", f"Expected one checkpoint, found {checkpoints}")
        else:
            ROUND_PATH = checkpoints[0].with_name("rounds.csv")
            record("application", "passed", f"seconds={time.perf_counter()-started:.2f}")
    else:
        RUN_OK = False
        record("application", "failed", f"exit={outcome.returncode}")
else:
    record("application", "skipped", "earlier gate failed")


## 4. Replay matched learners and audit chronological coverage


In [ ]:
import base64
AUDITOR_SOURCE = base64.b64decode("".join([
    'IiIiRVhQLTEwIGNhdXNhbCByZXBsYXkgYW5kIGVtcGlyaWNhbCBuZXh0LWJhdGNoIGludGVydmFsIGF1ZGl0LgoKVGhlIGlu',
    'dGVydmFsIHRhcmdldCBpcyB0aGUgKm9ic2VydmVkIG5leHQtYmF0Y2ggZW1waXJpY2FsIG9iamVjdGl2ZSBhdCB0aGUKY2hv',
    'c2VuIGFjdGlvbiouIFRoaXMgaXMgZGlzdGluY3QgZnJvbSBhbiB1bm9ic2VydmVkIHBvcHVsYXRpb24gb2JqZWN0aXZlLgpD',
    'aHJvbm9sb2dpY2FsIHJlc2lkdWFsIHF1YW50aWxlcyBhcmUgZGVzY3JpcHRpdmUgdW5kZXIgdGhpcyBkcmlmdGluZyBzdHJl',
    'YW07Cm5vIGV4Y2hhbmdlYWJpbGl0eSBvciB0aW1lLXVuaWZvcm0gY292ZXJhZ2UgZ3VhcmFudGVlIGlzIGFzc2VydGVkLgoi',
    'IiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBv',
    'cnQgc3lzCgppbXBvcnQgbnVtcHkgYXMgbnAKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2',
    'ZSgpLnBhcmVudHNbMV0gLyAic3JjIikpCmZyb20gaW50ZXJ2YWxfcGFyZXRvLmFsZ29yaXRobXMgaW1wb3J0IE1pZHBvaW50',
    'T0dECmZyb20gaW50ZXJ2YWxfcGFyZXRvLmFwcGxpY2F0aW9ucy5zdHJlYW1pbmcgaW1wb3J0IENsYXNzaWZpY2F0aW9uQmF0',
    'Y2gsIG9iamVjdGl2ZV9hbmRfc3ViZ3JhZGllbnRzCgoKUFJFRkVSRU5DRVMgPSB7CiAgICAiamNhbV84MF8xNV8wNSI6IG5w',
    'LmFycmF5KFswLjgwLCAwLjE1LCAwLjA1XSksCiAgICAibG9naXN0aWNfb25seSI6IG5wLmFycmF5KFsxLjAsIDAuMCwgMC4w',
    'XSksCiAgICAiZ2FwX2hlYXZ5XzY1XzMwXzA1IjogbnAuYXJyYXkoWzAuNjUsIDAuMzAsIDAuMDVdKSwKICAgICJyZXNvdXJj',
    'ZV9oZWF2eV83MF8xMF8yMCI6IG5wLmFycmF5KFswLjcwLCAwLjEwLCAwLjIwXSksCn0KCgpkZWYgX3F1YW50aWxlKHBhc3Q6',
    'IGxpc3RbZmxvYXRdLCBhbHBoYTogZmxvYXQpIC0+IGZsb2F0OgogICAgIyBTcGxpdC1jb25mb3JtYWwgb3JkZXIgc3RhdGlz',
    'dGljIGFwcGxpZWQgdG8gY2hyb25vbG9naWNhbCByZXNpZHVhbHMgYXMKICAgICMgYSAqaGV1cmlzdGljKi4gVGhlIG9yZGVy',
    'IHN0YXRpc3RpYyBhbG9uZSBkb2VzIG5vdCBnaXZlIHZhbGlkaXR5IGhlcmUuCiAgICBuID0gbGVuKHBhc3QpCiAgICByYW5r',
    'ID0gaW50KG5wLmNlaWwoKG4gKyAxKSAqICgxIC0gYWxwaGEpKSkKICAgIHJldHVybiBmbG9hdChucC5wYXJ0aXRpb24obnAu',
    'YXNhcnJheShwYXN0KSwgbWluKHJhbmssIG4pIC0gMSlbbWluKHJhbmssIG4pIC0gMV0pCgoKZGVmIGF1ZGl0KGRhdGFfcGF0',
    'aDogUGF0aCwgcm91bmRzX3BhdGg6IFBhdGgsIG91dHB1dDogUGF0aCwgZXRhOiBmbG9hdCA9IDAuMDUsCiAgICAgICAgICB3',
    'YXJtdXA6IGludCA9IDIwLCBhbHBoYTogZmxvYXQgPSAwLjEwKSAtPiBkaWN0OgogICAgaWYgZXRhIDw9IDAgb3Igd2FybXVw',
    'IDwgMSBvciBub3QgMCA8IGFscGhhIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIGV0YSwgd2FybXVw',
    'LCBvciBhbHBoYSIpCiAgICB3aXRoIG5wLmxvYWQoZGF0YV9wYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGRhdGE6CiAg',
    'ICAgICAgeCA9IG5wLmFzYXJyYXkoZGF0YVsiZmVhdHVyZXMiXSwgZmxvYXQpCiAgICAgICAgeSA9IG5wLmFzYXJyYXkoZGF0',
    'YVsibGFiZWxzIl0sIGZsb2F0KQogICAgICAgIGcgPSBucC5hc2FycmF5KGRhdGFbImdyb3VwcyJdLCBpbnQpCiAgICAgICAg',
    'c3VwcGxpZWRfcmFkaWkgPSBucC5hc2FycmF5KGRhdGFbInJhZGlpIl0sIGZsb2F0KQogICAgd2l0aCByb3VuZHNfcGF0aC5v',
    'cGVuKG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICBvcmlnaW5hbCA9IGxpc3QoY3N2',
    'LkRpY3RSZWFkZXIoaGFuZGxlKSkKICAgIGlmIGxlbihvcmlnaW5hbCkgIT0gbGVuKHgpIG9yIHdhcm11cCA+PSBsZW4oeCk6',
    'CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicm91bmQgY291bnQgbWlzbWF0Y2ggb3Igd2FybXVwIGV4Y2VlZHMgc3RyZWFt',
    'IikKCiAgICBsZWFybmVycyA9IHtuYW1lOiBNaWRwb2ludE9HRCh4LnNoYXBlWzJdLCBldGEpIGZvciBuYW1lIGluIFBSRUZF',
    'UkVOQ0VTfQogICAgcmVzaWR1YWxzOiBsaXN0W2xpc3RbZmxvYXRdXSA9IFtbXSwgW10sIFtdXQogICAgcm93czogbGlzdFtk',
    'aWN0XSA9IFtdCiAgICBtYXhfcmVwbGF5X2FjdGlvbl9lcnJvciA9IDAuMAogICAgZm9yIHQgaW4gcmFuZ2UobGVuKHgpKToK',
    'ICAgICAgICBiYXRjaCA9IENsYXNzaWZpY2F0aW9uQmF0Y2goeFt0XSwgeVt0XSwgZ1t0XSkKICAgICAgICBmb3IgbmFtZSwg',
    'bGVhcm5lciBpbiBsZWFybmVycy5pdGVtcygpOgogICAgICAgICAgICBhY3Rpb24gPSBsZWFybmVyLmFjdCgpCiAgICAgICAg',
    'ICAgIHZhbHVlcywgZ3JhZGllbnRzID0gb2JqZWN0aXZlX2FuZF9zdWJncmFkaWVudHMoYmF0Y2gsIGFjdGlvbikKICAgICAg',
    'ICAgICAgc2NvcmVzID0geFt0XSBAIGFjdGlvbgogICAgICAgICAgICBwcm9iYWJpbGl0eSA9IDEgLyAoMSArIG5wLmV4cCgt',
    'bnAuY2xpcChzY29yZXMsIC03MDAsIDcwMCkpKQogICAgICAgICAgICByb3cgPSB7CiAgICAgICAgICAgICAgICAiYmF0Y2gi',
    'OiB0ICsgMSwgIm1ldGhvZCI6IG5hbWUsCiAgICAgICAgICAgICAgICAibG9nbG9zcyI6IGZsb2F0KHZhbHVlc1swXSksICJz',
    'Y29yZV9nYXAiOiBmbG9hdCh2YWx1ZXNbMV0pLAogICAgICAgICAgICAgICAgInJlc291cmNlIjogZmxvYXQodmFsdWVzWzJd',
    'KSwKICAgICAgICAgICAgICAgICJhY2N1cmFjeSI6IGZsb2F0KG5wLm1lYW4oKHNjb3JlcyA+PSAwKSA9PSAoeVt0XSA+IDAp',
    'KSksCiAgICAgICAgICAgICAgICAiYnJpZXIiOiBmbG9hdChucC5tZWFuKChwcm9iYWJpbGl0eSAtICh5W3RdID4gMCkpICoq',
    'IDIpKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBpZiBuYW1lID09ICJqY2FtXzgwXzE1XzA1IjoKICAgICAgICAgICAg',
    'ICAgIGxvZ2dlZCA9IG5wLmFzYXJyYXkoanNvbi5sb2FkcyhvcmlnaW5hbFt0XVsiYWN0aW9uIl0pLCBmbG9hdCkKICAgICAg',
    'ICAgICAgICAgIG1heF9yZXBsYXlfYWN0aW9uX2Vycm9yID0gbWF4KG1heF9yZXBsYXlfYWN0aW9uX2Vycm9yLCBmbG9hdChu',
    'cC5tYXgobnAuYWJzKGFjdGlvbiAtIGxvZ2dlZCkpKSkKICAgICAgICAgICAgICAgIGlmIG1heF9yZXBsYXlfYWN0aW9uX2Vy',
    'cm9yID4gMWUtOToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRGVmYXVsdCByZXBsYXkgZGlmZmVy',
    'cyBmcm9tIGxvZ2dlZCBhY3Rpb24gYXQgYmF0Y2gge3QgKyAxfSIpCiAgICAgICAgICAgICAgICBpZiB0ID4gMDoKICAgICAg',
    'ICAgICAgICAgICAgICBwcmV2aW91cyA9IENsYXNzaWZpY2F0aW9uQmF0Y2goeFt0IC0gMV0sIHlbdCAtIDFdLCBnW3QgLSAx',
    'XSkKICAgICAgICAgICAgICAgICAgICBmb3JlY2FzdCwgXyA9IG9iamVjdGl2ZV9hbmRfc3ViZ3JhZGllbnRzKHByZXZpb3Vz',
    'LCBhY3Rpb24pCiAgICAgICAgICAgICAgICAgICAgZXJyb3IgPSBucC5hYnModmFsdWVzIC0gZm9yZWNhc3QpCiAgICAgICAg',
    'ICAgICAgICAgICAgcm93LnVwZGF0ZSh7ZiJmb3JlY2FzdF97an0iOiBmbG9hdChmb3JlY2FzdFtqXSkgZm9yIGogaW4gcmFu',
    'Z2UoMyl9KQogICAgICAgICAgICAgICAgICAgIHJvdy51cGRhdGUoe2YiZXJyb3Jfe2p9IjogZmxvYXQoZXJyb3Jbal0pIGZv',
    'ciBqIGluIHJhbmdlKDMpfSkKICAgICAgICAgICAgICAgICAgICBpZiB0ID49IHdhcm11cDoKICAgICAgICAgICAgICAgICAg',
    'ICAgICAgcmFkaXVzID0gbnAuYXJyYXkoW19xdWFudGlsZShyZXNpZHVhbHNbal0sIGFscGhhKSBmb3IgaiBpbiByYW5nZSgz',
    'KV0pCiAgICAgICAgICAgICAgICAgICAgICAgIHJvdy51cGRhdGUoe2YicmFkaXVzX3tqfSI6IGZsb2F0KHJhZGl1c1tqXSkg',
    'Zm9yIGogaW4gcmFuZ2UoMyl9KQogICAgICAgICAgICAgICAgICAgICAgICByb3cudXBkYXRlKHtmImNvdmVyZWRfe2p9Ijog',
    'aW50KGVycm9yW2pdIDw9IHJhZGl1c1tqXSArIDFlLTEyKSBmb3IgaiBpbiByYW5nZSgzKX0pCiAgICAgICAgICAgICAgICAg',
    'ICAgICAgIHJvdy51cGRhdGUoe2YiZml4ZWRfY292ZXJlZF97an0iOiBpbnQoZXJyb3Jbal0gPD0gc3VwcGxpZWRfcmFkaWlb',
    'dCwgal0gKyAxZS0xMikgZm9yIGogaW4gcmFuZ2UoMyl9KQogICAgICAgICAgICAgICAgICAgICAgICByb3dbImFsbF9jb3Zl',
    'cmVkIl0gPSBpbnQoYm9vbChucC5hbGwoZXJyb3IgPD0gcmFkaXVzICsgMWUtMTIpKSkKICAgICAgICAgICAgICAgICAgICBm',
    'b3IgaiBpbiByYW5nZSgzKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzaWR1YWxzW2pdLmFwcGVuZChmbG9hdChlcnJv',
    'cltqXSkpCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHJvdykKICAgICAgICAgICAgbGVhcm5lci5vYnNlcnZlX2dyYWRpZW50',
    'KFBSRUZFUkVOQ0VTW25hbWVdIEAgZ3JhZGllbnRzKQoKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29r',
    'PVRydWUpCiAgICBmaWVsZHMgPSBzb3J0ZWQoe2tleSBmb3Igcm93IGluIHJvd3MgZm9yIGtleSBpbiByb3d9KQogICAgd2l0',
    'aCAob3V0cHV0IC8gInJlcGxheV9hbmRfaW50ZXJ2YWxzLmNzdiIpLm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0i',
    'dXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoaGFuZGxlLCBmaWVsZG5hbWVzPWZp',
    'ZWxkcykKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIG1l',
    'dGhvZHMgPSB7fQogICAgZm9yIG5hbWUgaW4gUFJFRkVSRU5DRVM6CiAgICAgICAgc3Vic2V0ID0gW3JvdyBmb3Igcm93IGlu',
    'IHJvd3MgaWYgcm93WyJtZXRob2QiXSA9PSBuYW1lXQogICAgICAgIG1ldGhvZHNbbmFtZV0gPSB7bWV0cmljOiBmbG9hdChu',
    'cC5tZWFuKFtyb3dbbWV0cmljXSBmb3Igcm93IGluIHN1YnNldF0pKQogICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG1l',
    'dHJpYyBpbiAoImxvZ2xvc3MiLCAic2NvcmVfZ2FwIiwgInJlc291cmNlIiwgImFjY3VyYWN5IiwgImJyaWVyIil9CiAgICBj',
    'YWxpYnJhdGVkID0gW3JvdyBmb3Igcm93IGluIHJvd3MgaWYgcm93WyJtZXRob2QiXSA9PSAiamNhbV84MF8xNV8wNSIgYW5k',
    'ICJhbGxfY292ZXJlZCIgaW4gcm93XQogICAgc3VtbWFyeSA9IHsKICAgICAgICAidGFyZ2V0IjogIm5leHQgb2JzZXJ2ZWQg',
    'YmF0Y2ggZW1waXJpY2FsIG9iamVjdGl2ZXMgYXQgdGhlIGN1cnJlbnQgYWN0aW9uIiwKICAgICAgICAiZm9yZWNhc3QiOiAi',
    'cHJldmlvdXMgb2JzZXJ2ZWQgYmF0Y2ggb2JqZWN0aXZlcyBldmFsdWF0ZWQgYXQgdGhlIGN1cnJlbnQgYWN0aW9uIiwKICAg',
    'ICAgICAicmFkaXVzX3J1bGUiOiAiZXhwYW5kaW5nIHBhc3QgYWJzb2x1dGUgZm9yZWNhc3QtZXJyb3IgOTB0aCBwZXJjZW50',
    'aWxlIG9yZGVyIHN0YXRpc3RpYyIsCiAgICAgICAgIm5vbWluYWxfbWFyZ2luYWxfY292ZXJhZ2UiOiAxIC0gYWxwaGEsCiAg',
    'ICAgICAgIndhcm11cF9iYXRjaGVzIjogd2FybXVwLAogICAgICAgICJldmFsdWF0ZWRfYmF0Y2hlcyI6IGxlbihjYWxpYnJh',
    'dGVkKSwKICAgICAgICAiZ3VhcmFudGVlZF9wb3B1bGF0aW9uX29yX3NlcXVlbnRpYWxfY292ZXJhZ2UiOiBGYWxzZSwKICAg',
    'ICAgICAibWV0aG9kcyI6IG1ldGhvZHMsCiAgICAgICAgIm1heF9yZXBsYXlfYWN0aW9uX2Vycm9yIjogbWF4X3JlcGxheV9h',
    'Y3Rpb25fZXJyb3IsCiAgICAgICAgImNhbGlicmF0aW9uIjogewogICAgICAgICAgICBmIm9iamVjdGl2ZV97an0iOiB7CiAg',
    'ICAgICAgICAgICAgICAiZW1waXJpY2FsX2NvdmVyYWdlIjogZmxvYXQobnAubWVhbihbcm93W2YiY292ZXJlZF97an0iXSBm',
    'b3Igcm93IGluIGNhbGlicmF0ZWRdKSksCiAgICAgICAgICAgICAgICAiZml4ZWRfYmFuZF9jb3ZlcmFnZSI6IGZsb2F0KG5w',
    'Lm1lYW4oW3Jvd1tmImZpeGVkX2NvdmVyZWRfe2p9Il0gZm9yIHJvdyBpbiBjYWxpYnJhdGVkXSkpLAogICAgICAgICAgICAg',
    'ICAgIm1lYW5fcmFkaXVzIjogZmxvYXQobnAubWVhbihbcm93W2YicmFkaXVzX3tqfSJdIGZvciByb3cgaW4gY2FsaWJyYXRl',
    'ZF0pKSwKICAgICAgICAgICAgICAgICJtZWFuX2Fic29sdXRlX2ZvcmVjYXN0X2Vycm9yIjogZmxvYXQobnAubWVhbihbcm93',
    'W2YiZXJyb3Jfe2p9Il0gZm9yIHJvdyBpbiBjYWxpYnJhdGVkXSkpLAogICAgICAgICAgICB9IGZvciBqIGluIHJhbmdlKDMp',
    'CiAgICAgICAgfSwKICAgICAgICAiam9pbnRfZW1waXJpY2FsX2NvdmVyYWdlIjogZmxvYXQobnAubWVhbihbcm93WyJhbGxf',
    'Y292ZXJlZCJdIGZvciByb3cgaW4gY2FsaWJyYXRlZF0pKSwKICAgIH0KICAgIChvdXRwdXQgLyAiYXVkaXQuanNvbiIpLndy',
    'aXRlX3RleHQoanNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gc3Vt',
    'bWFyeQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigp',
    'CiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIu',
    'YWRkX2FyZ3VtZW50KCItLXJvdW5kcyIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1l',
    'bnQoIi0tb3V0cHV0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkK',
    'ICAgIHByaW50KGpzb24uZHVtcHMoYXVkaXQoYXJncy5kYXRhLCBhcmdzLnJvdW5kcywgYXJncy5vdXRwdXQpLCBpbmRlbnQ9',
    'MikpCg==',
]))
(CODE / "experiments" / "audit_bank_stream.py").write_bytes(AUDITOR_SOURCE)
if RUN_OK:
    command = [sys.executable, str(CODE / "experiments" / "audit_bank_stream.py"),
               "--data", str(DATA), "--rounds", str(ROUND_PATH),
               "--output", str(RESULTS / "audit")]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-9000:])
    if outcome.returncode == 0:
        AUDIT = json.loads((RESULTS / "audit" / "audit.json").read_text(encoding="utf-8"))
        record("audit", "passed", f"evaluated={AUDIT['evaluated_batches']} batches")
    else:
        RUN_OK = False
        record("audit", "failed", f"exit={outcome.returncode}")
else:
    record("audit", "skipped", "earlier gate failed")


## 5. Zip the complete audit output


In [ ]:
import zipfile
(RESULTS / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
if BOOTSTRAP_ERROR is None:
    shutil.copy2(SOURCE_META, RESULTS / "bank_stream.metadata.json")
FINAL_ZIP = WORK_ROOT / "result_jcam_exp10_audit.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=(Path("exp10_bank_audit") / path.relative_to(RESULTS)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
